# 🛡️ Credit Card Fraud Detection — Complete ML Pipeline

---

## 📋 Table of Contents

| Step | Topic | Description |
|------|-------|-------------|
| **1** | Data Generation | Create synthetic credit card dataset |
| **2** | EDA | Exploratory Data Analysis — understand the data |
| **3** | Preprocessing | Clean, scale, split, and balance with SMOTE |
| **4** | Anomaly Detection | Isolation Forest & Local Outlier Factor |
| **5** | XGBoost Classifier | Train supervised gradient boosting model |
| **6** | Evaluation | ROC curves, confusion matrices, comparison |
| **7** | Prediction Demo | Test on new transactions |

---

**Tools Used:** Python, Pandas, Scikit-Learn, XGBoost, Imbalanced-Learn, Matplotlib, Seaborn

In [ ]:
# ══════════════════════════════════════════════════════════════
# IMPORTS — All libraries we'll need
# ══════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_curve, auc, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.style.use('dark_background')
sns.set_palette(['#6C63FF', '#FF6584', '#2ECC71', '#F1C40F', '#E74C3C'])
print('✅ All imports successful!')

---
## 📦 Step 1: Generate the Dataset

We generate a **synthetic dataset** that mirrors the structure of the [Kaggle Credit Card Fraud Detection dataset](https://www.kaggle.com/mlg-ulb/creditcardfraud):

| Column | Description |
|--------|-------------|
| `Time` | Seconds elapsed from first transaction |
| `V1–V28` | 28 PCA-transformed features (anonymized) |
| `Amount` | Transaction amount in USD |
| `Class` | **0** = Legitimate, **1** = Fraud |

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 1: Generate Synthetic Dataset
# ══════════════════════════════════════════════════════════════
import sys, os
sys.path.insert(0, os.getcwd())

from data.download_data import generate_dataset

# Generate the dataset (saved to data/creditcard.csv)
df = generate_dataset()
print(f'\n📊 Dataset shape: {df.shape}')
df.head()

---
## 🔍 Step 2: Exploratory Data Analysis (EDA)

Before building models, we need to **understand our data**:
- How imbalanced are the classes?
- What do transaction amounts look like?
- Are there patterns in time?

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 2a: Class Distribution — How imbalanced is the data?
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
class_counts = df['Class'].value_counts()
colors = ['#6C63FF', '#FF6584']
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts.values, color=colors, 
            edgecolor='white', linewidth=0.5)
axes[0].set_title('Class Distribution (Count)', fontsize=14, fontweight='bold', pad=15)
axes[0].set_ylabel('Number of Transactions')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Legitimate', 'Fraud'], colors=colors,
            autopct='%1.2f%%', startangle=90, explode=(0, 0.1),
            textprops={'fontsize': 12, 'color': 'white'},
            wedgeprops={'edgecolor': '#0E1117', 'linewidth': 2})
axes[1].set_title('Class Distribution (%)', fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print(f'\n📊 Fraud rate: {df["Class"].mean()*100:.2f}%')
print(f'   This extreme imbalance is why we need SMOTE!')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 2b: Transaction Amount Analysis
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by class
axes[0].hist(df[df['Class']==0]['Amount'], bins=50, alpha=0.7, color='#6C63FF', 
             label='Legitimate', edgecolor='none')
axes[0].hist(df[df['Class']==1]['Amount'], bins=50, alpha=0.7, color='#FF6584', 
             label='Fraud', edgecolor='none')
axes[0].set_title('Transaction Amount Distribution', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Amount ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot comparison
bp = axes[1].boxplot([df[df['Class']==0]['Amount'], df[df['Class']==1]['Amount']],
                     labels=['Legitimate', 'Fraud'], patch_artist=True,
                     boxprops=dict(facecolor='#6C63FF', alpha=0.7),
                     medianprops=dict(color='white', linewidth=2))
bp['boxes'][1].set_facecolor('#FF6584')
axes[1].set_title('Amount by Class (Box Plot)', fontsize=14, fontweight='bold', pad=15)
axes[1].set_ylabel('Amount ($)')

plt.tight_layout()
plt.show()

print(f'\n💰 Amount Statistics:')
print(f'   Legitimate — Mean: ${df[df["Class"]==0]["Amount"].mean():.2f}, '
      f'Max: ${df[df["Class"]==0]["Amount"].max():.2f}')
print(f'   Fraud      — Mean: ${df[df["Class"]==1]["Amount"].mean():.2f}, '
      f'Max: ${df[df["Class"]==1]["Amount"].max():.2f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 2c: Correlation Heatmap (Top Features)
# ══════════════════════════════════════════════════════════════
# Show correlation of top features with Class
correlations = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = correlations.head(10).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df[top_features + ['Class']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdPu',
            linewidths=1, linecolor='#0E1117', cbar_kws={'shrink': 0.8},
            annot_kws={'size': 10})
ax.set_title('Correlation Heatmap (Top Features vs Class)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print(f'\n🔝 Top 5 correlated features with fraud:')
for feat, corr in correlations.head(5).items():
    print(f'   {feat}: {corr:.4f}')

---
## ⚙️ Step 3: Data Preprocessing

### Pipeline Steps:
1. **Scale** `Amount` and `Time` using `StandardScaler` (V1-V28 are already PCA-scaled)
2. **Split** into 80% train / 20% test with stratification
3. **Balance** training data with SMOTE (Synthetic Minority Oversampling)

> ⚠️ **SMOTE is applied ONLY to training data!** Test data stays imbalanced to reflect real-world performance.

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 3a: Scale Features
# ══════════════════════════════════════════════════════════════
scaler = StandardScaler()

df_processed = df.copy()
df_processed['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df_processed['Time_Scaled'] = StandardScaler().fit_transform(df[['Time']])
df_processed = df_processed.drop(['Amount', 'Time'], axis=1)

print('✅ Features scaled:')
print(f'   Amount_Scaled: mean={df_processed["Amount_Scaled"].mean():.4f}, std={df_processed["Amount_Scaled"].std():.4f}')
print(f'   Time_Scaled:   mean={df_processed["Time_Scaled"].mean():.4f}, std={df_processed["Time_Scaled"].std():.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 3b: Train/Test Split (Stratified)
# ══════════════════════════════════════════════════════════════
X = df_processed.drop('Class', axis=1)
y = df_processed['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Data split:')
print(f'   Training: {X_train.shape[0]:,} samples (Fraud: {(y_train==1).sum()})')
print(f'   Testing:  {X_test.shape[0]:,} samples (Fraud: {(y_test==1).sum()})')
print(f'   Features: {X_train.shape[1]}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 3c: SMOTE — Balance the Training Data
# ══════════════════════════════════════════════════════════════
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Visualize before vs after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before SMOTE
before = y_train.value_counts()
axes[0].bar(['Legitimate', 'Fraud'], before.values, color=['#6C63FF', '#FF6584'],
            edgecolor='white', linewidth=0.5)
axes[0].set_title('Before SMOTE', fontsize=14, fontweight='bold')
for i, v in enumerate(before.values):
    axes[0].text(i, v+20, f'{v:,}', ha='center', fontsize=12, fontweight='bold')

# After SMOTE
after = pd.Series(y_train_smote).value_counts()
axes[1].bar(['Legitimate', 'Fraud'], after.values, color=['#6C63FF', '#FF6584'],
            edgecolor='white', linewidth=0.5)
axes[1].set_title('After SMOTE', fontsize=14, fontweight='bold')
for i, v in enumerate(after.values):
    axes[1].text(i, v+20, f'{v:,}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n✅ SMOTE applied! Training data is now balanced:')
print(f'   Class 0: {(y_train_smote==0).sum():,}')
print(f'   Class 1: {(y_train_smote==1).sum():,}')

---
## 🌲 Step 4: Anomaly Detection

### 4a. Isolation Forest
**How it works:** Builds random trees that try to "isolate" each data point. Anomalies (fraud) get isolated in fewer splits because they're different from the majority.

### 4b. Local Outlier Factor (LOF)
**How it works:** Measures the local density around each point. Points in sparse regions (far from neighbors) are flagged as outliers.

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 4a: Isolation Forest
# ══════════════════════════════════════════════════════════════
print('🌲 Training Isolation Forest...')

# Using UNBALANCED training data (anomaly detection doesn't need SMOTE)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.017,  # ~1.7% expected outliers
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train)

# Predict: -1 = outlier (fraud), 1 = inlier (legit)
iso_pred_raw = iso_forest.predict(X_test)
iso_pred = np.where(iso_pred_raw == -1, 1, 0)  # Convert to 0/1
iso_scores = iso_forest.decision_function(X_test)

print(f'\n✅ Isolation Forest Results:')
print(f'   Accuracy: {accuracy_score(y_test, iso_pred):.4f}')
print(f'\n{classification_report(y_test, iso_pred, target_names=["Legitimate", "Fraud"], zero_division=0)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 4b: Local Outlier Factor (LOF)
# ══════════════════════════════════════════════════════════════
print('🔍 Training Local Outlier Factor...')

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.017,
    novelty=True,  # Enable predict on new data
    n_jobs=-1
)
lof.fit(X_train)

# Predict
lof_pred_raw = lof.predict(X_test)
lof_pred = np.where(lof_pred_raw == -1, 1, 0)
lof_scores = lof.decision_function(X_test)

print(f'\n✅ LOF Results:')
print(f'   Accuracy: {accuracy_score(y_test, lof_pred):.4f}')
print(f'\n{classification_report(y_test, lof_pred, target_names=["Legitimate", "Fraud"], zero_division=0)}')

---
## 🚀 Step 5: XGBoost Classifier

**XGBoost** (eXtreme Gradient Boosting) is the gold standard for tabular data classification:
1. Starts with a weak model
2. Builds new trees that focus on previous errors
3. Each tree corrects the mistakes of the ensemble
4. Uses `scale_pos_weight` to handle class imbalance

> We train on **SMOTE-balanced** data for better fraud detection.

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 5: Train XGBoost Classifier
# ══════════════════════════════════════════════════════════════
print('🚀 Training XGBoost Classifier...')

# Calculate class weight for imbalance handling
scale_weight = (y_train_smote == 0).sum() / max((y_train_smote == 1).sum(), 1)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0
)

xgb_model.fit(
    X_train_smote, y_train_smote,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Predictions
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f'\n✅ XGBoost Results:')
print(f'   Accuracy: {accuracy_score(y_test, xgb_pred):.4f}')
print(f'   Precision: {precision_score(y_test, xgb_pred, zero_division=0):.4f}')
print(f'   Recall: {recall_score(y_test, xgb_pred, zero_division=0):.4f}')
print(f'   F1 Score: {f1_score(y_test, xgb_pred, zero_division=0):.4f}')
print(f'\n{classification_report(y_test, xgb_pred, target_names=["Legitimate", "Fraud"], zero_division=0)}')

---
## 📈 Step 6: Evaluation & Visualization

### Key Metrics Explained:
- **ROC Curve**: True Positive Rate vs False Positive Rate (closer to top-left = better)
- **AUC**: Area Under the ROC Curve (1.0 = perfect, 0.5 = random)
- **Confusion Matrix**: Shows exactly where the model gets it right/wrong
- **Recall (Sensitivity)**: Most important for fraud — "did we catch the fraud?"

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 6a: ROC Curves — All Three Models
# ══════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 7))

# XGBoost ROC
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)
auc_xgb = auc(fpr_xgb, tpr_xgb)
ax.plot(fpr_xgb, tpr_xgb, color='#6C63FF', linewidth=2.5,
        label=f'XGBoost (AUC = {auc_xgb:.4f})')
ax.fill_between(fpr_xgb, tpr_xgb, alpha=0.1, color='#6C63FF')

# Isolation Forest ROC
fpr_iso, tpr_iso, _ = roc_curve(y_test, -iso_scores)  # Negate scores
auc_iso = auc(fpr_iso, tpr_iso)
ax.plot(fpr_iso, tpr_iso, color='#FF6584', linewidth=2.5,
        label=f'Isolation Forest (AUC = {auc_iso:.4f})')

# LOF ROC
fpr_lof, tpr_lof, _ = roc_curve(y_test, -lof_scores)
auc_lof = auc(fpr_lof, tpr_lof)
ax.plot(fpr_lof, tpr_lof, color='#2ECC71', linewidth=2.5,
        label=f'LOF (AUC = {auc_lof:.4f})')

# Random baseline
ax.plot([0, 1], [0, 1], 'w--', linewidth=1, alpha=0.5, label='Random (AUC = 0.50)')

ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curve Comparison — All Models', fontsize=16, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f'\n📊 AUC Summary:')
print(f'   XGBoost:          {auc_xgb:.4f}')
print(f'   Isolation Forest: {auc_iso:.4f}')
print(f'   LOF:              {auc_lof:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 6b: Confusion Matrices — All Three Models
# ══════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = {
    'XGBoost': xgb_pred,
    'Isolation Forest': iso_pred,
    'LOF': lof_pred
}

for idx, (name, preds) in enumerate(models.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
                xticklabels=['Legitimate', 'Fraud'],
                yticklabels=['Legitimate', 'Fraud'],
                linewidths=2, linecolor='#0E1117',
                annot_kws={'size': 14, 'fontweight': 'bold'},
                ax=axes[idx])
    axes[idx].set_title(f'{name}', fontsize=14, fontweight='bold', pad=15)
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 6c: Feature Importance (XGBoost)
# ══════════════════════════════════════════════════════════════
importance = dict(zip(X_train.columns, xgb_model.feature_importances_))
sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:15]

names = [f[0] for f in sorted_imp][::-1]
scores = [f[1] for f in sorted_imp][::-1]

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.RdPu(np.linspace(0.3, 0.9, len(names)))
ax.barh(names, scores, color=colors, edgecolor='none', height=0.6)
ax.set_xlabel('Importance Score', fontsize=13)
ax.set_title('Top 15 Feature Importance (XGBoost)', fontsize=16, fontweight='bold', pad=15)
ax.grid(True, axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

print(f'\n🏆 Top 5 Most Important Features:')
for name, score in sorted_imp[:5]:
    print(f'   {name}: {score:.4f}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 6d: Model Comparison Summary Table
# ══════════════════════════════════════════════════════════════
comparison = pd.DataFrame({
    'Model': ['XGBoost', 'Isolation Forest', 'LOF'],
    'Type': ['Supervised', 'Unsupervised', 'Unsupervised'],
    'Accuracy': [
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, iso_pred),
        accuracy_score(y_test, lof_pred)
    ],
    'Precision': [
        precision_score(y_test, xgb_pred, zero_division=0),
        precision_score(y_test, iso_pred, zero_division=0),
        precision_score(y_test, lof_pred, zero_division=0)
    ],
    'Recall': [
        recall_score(y_test, xgb_pred, zero_division=0),
        recall_score(y_test, iso_pred, zero_division=0),
        recall_score(y_test, lof_pred, zero_division=0)
    ],
    'F1 Score': [
        f1_score(y_test, xgb_pred, zero_division=0),
        f1_score(y_test, iso_pred, zero_division=0),
        f1_score(y_test, lof_pred, zero_division=0)
    ],
    'AUC': [auc_xgb, auc_iso, auc_lof]
})

# Format numeric columns
for col in ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']:
    comparison[col] = comparison[col].round(4)

print('\n📋 MODEL COMPARISON SUMMARY')
print('=' * 80)
print(comparison.to_string(index=False))
print('=' * 80)

best_model = comparison.loc[comparison['AUC'].idxmax(), 'Model']
print(f'\n🏆 Best model by AUC: {best_model}')

---
## 🔮 Step 7: Prediction Demo

Test the trained XGBoost model on sample transactions to see predictions in action.

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 7: Test Predictions on Sample Transactions
# ══════════════════════════════════════════════════════════════
print('🔮 Prediction Demo — Testing on 5 random transactions\n')

np.random.seed(42)
sample_indices = np.random.choice(len(X_test), 5, replace=False)

for i, idx in enumerate(sample_indices):
    sample = X_test.iloc[idx:idx+1]
    true_label = y_test.iloc[idx]
    
    pred = xgb_model.predict(sample)[0]
    proba = xgb_model.predict_proba(sample)[0]
    
    correct = '✅' if pred == true_label else '❌'
    label = '🔴 FRAUD' if pred == 1 else '🟢 LEGIT'
    true = '🔴 FRAUD' if true_label == 1 else '🟢 LEGIT'
    
    print(f'  Transaction #{i+1}:')
    print(f'    True:      {true}')
    print(f'    Predicted: {label} (Confidence: {max(proba)*100:.1f}%) {correct}')
    print(f'    Fraud Probability: {proba[1]*100:.2f}%')
    print()

In [ ]:
# ══════════════════════════════════════════════════════════════
# STEP 7b: Save the trained model for the Streamlit app
# ══════════════════════════════════════════════════════════════
import joblib

os.makedirs('saved_models', exist_ok=True)
joblib.dump(xgb_model, 'saved_models/xgboost_model.joblib')
joblib.dump(iso_forest, 'saved_models/isolation_forest.joblib')
joblib.dump(lof, 'saved_models/lof_model.joblib')
joblib.dump(scaler, 'saved_models/scaler.joblib')

print('✅ All models saved to saved_models/ directory!')
print('\n🚀 To launch the web dashboard, run:')
print('   streamlit run app.py')

---
## ✅ Summary

### What we accomplished:

| Step | What | Status |
|------|------|--------|
| 1 | Generated synthetic credit card dataset | ✅ |
| 2 | Performed EDA (class distribution, amounts, correlations) | ✅ |
| 3 | Preprocessed data (scaled, split, balanced with SMOTE) | ✅ |
| 4 | Trained Isolation Forest & LOF anomaly detectors | ✅ |
| 5 | Trained XGBoost supervised classifier | ✅ |
| 6 | Evaluated all models (ROC curves, confusion matrices) | ✅ |
| 7 | Demonstrated predictions & saved models | ✅ |

### Key Takeaways:
- **XGBoost** performs best overall (highest AUC) — supervised learning wins!
- **Isolation Forest & LOF** are useful when labels are unavailable
- **SMOTE** significantly improved fraud recall
- **Recall** is the most important metric for fraud detection (catching all frauds)

### Next Steps:
- Launch the **Streamlit dashboard**: `streamlit run app.py`
- Try different hyperparameters for better performance
- Deploy to a cloud platform (Streamlit Cloud, Heroku, etc.)